# Data preprocessing


## Table of contents

1. [**enc2017 corpus**](#enc2017_corpus)
   1. [Vabamorf's morphological tagging](#vabamorf_morf_tag_1)
   2. [Analysis of the results](#analysis_of_the_results_1)
   3. [Further preprocessing](#further_preprocessing_1)
2. [**Estonian Universal Dependencies' EDT corpus**](#estonian_ud_edt_corpus)
   1. [Extracting the data](#extracting_the_data_2)
   2. [Gathering the data](#gathering_the_data_2)
   3. [Analysis of the results](#analysis_of_the_results_2)
   4. [Further preprocessing](#further_preprocessing_2)
3. [**Homonym disambiguation corpus**](#homonym_disambiguation_corpus)
   1. [LabelStudio JSON files](#labelstudio_json_files_3)
      1. [Extracting the data](#extracting_the_data_31)
      2. [Gathering the data](#gathering_the_data_31)
      3. [Analysis of the results](#analysis_of_the_results_31)
      4. [Further preprocessing](#further_preprocessing_31)
   2. [JSONL file](#jsonl_file_3)
      1. [Gathering the data](#gathering_the_data_32)
      2. [Further preprocessing](#further_preprocessing_32)
4. [**Model related preprocessing**](#model_related_preprocessing_4)

[End](#end)


In [1]:
print("Test")

Test


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

<a id='enc2017_corpus'></a>


## enc2017 corpus

Data has been gathered from the [corpus](https://github.com/estnltk/eval_experiments_lrec_2020/blob/master/scripts_and_data/enc2017_selection_plain_texts_json.zip) that was used in evaluation experiments reported in LREC 2020 paper "EstNLTK 1.6: Remastered Estonian NLP Pipeline"[^1].

[^1]: Sven Laur, Siim Orasmaa, Dage Särg, Paul Tammo. "EstNLTK 1.6: Remastered Estonian NLP Pipeline". _Proceedings of The 12th Language Resources and Evaluation Conference_. European Language Resources Association: Marseille, France, May 2020, p. 7154-7162


<a id='vabamorf_morf_tag_1'></a>


### Vabamorf's morphological tagging


In [3]:
import numpy as np
import pandas as pd
import estnltk
import tqdm
import json

from scripts.preprocessing.enc2017.preprocess import (
    Preprocessor as Enc2017Preprocessor,
)

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
jsons_dir = ENC2017_DIRS["raw"] / "_plain_texts_json"
jsons = list(jsons_dir.glob("*.json"))

In [4]:
Enc2017Preprocessor.create_json_file_by_file_enc2017(
    jsons=jsons,
    save_dir=ENC2017_DIRS["processed"] / "_vabamorf_morph_texts_json",
    do_morph_layer=True,
    bert_morph_tagger=None,
    necessary_layers=["words", "sentences", "morph_analysis"],
    ignore_errors=False,
    replace_files=False,
)

Beginning to morphologically tag file by file


100%|██████████| 18486/18486 [00:02<00:00, 6677.21it/s]

Morphological tagging completed successfully


In [4]:
jsons_dir = ENC2017_DIRS["processed"] / "_vabamorf_morph_texts_json"
jsons = list(jsons_dir.glob("*.json"))

In [5]:
text = estnltk.converters.json_to_text(file=jsons[0])

print("morph_analysis" in text.layers)

True


In [6]:
text.sentences[0].morph_analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Teisipäeval', [{'normalized_text': 'Teisipäeval', 'lemma': 'teisipäev', 'root': 'teisi_päev', 'root_tokens': ['teisi', 'päev'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span('saabus', [{'normalized_text': 'saabus', 'lemma': 'saabuma', 'root': 'saabu', 'root_tokens': ['saabu'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('Iisraelist', [{'normalized_text': 'Iisraelist', 'lemma': 'Iisrael', 'root': 'Iisrael', 'root_tokens': ['Iisrael'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'H'}]),
Span('Tallinna', [{'normalized_text': 'Tallinna', 'lemma': 'Tallinn', 'root': 'Tallinn', 'root_tokens': ['Tallinn'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('lennujaama', [{'normalized_text': 'lennujaama', 'lemma': 'lennujaam', 'root': 'lennu_jaam', 'root_tokens': ['lennu', 'jaam'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}]),
Span('42aastane', [{'normalized_text': '42aastane', 'lemma': '42aastane', 'root': '42_aastane', 'root_tokens': ['42', 'aastane'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('Avraham', [{'normalized_text': 'Avraham', 'lemma': 'Avraham', 'root': 'Avraham', 'root_tokens': ['Avraham'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('kes', [{'normalized_text': 'kes', 'lemma': 'kes', 'root': 'kes', 'root_tokens': ['kes'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}, {'normalized_text': 'kes', 'lemma': 'kes', 'root': 'kes', 'root_tokens': ['kes'], 'ending': '0', 'clitic': '', 'form': 'pl n', 'partofspeech': 'P'}]),
Span('pidi', [{'normalized_text': 'pidi', 'lemma': 'pidama', 'root': 'pida', 'root_tokens': ['pida'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('hiljem', [{'normalized_text': 'hiljem', 'lemma': 'hiljem', 'root': 'hiljem', 'root_tokens': ['hiljem'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('transiiditsoonist', [{'normalized_text': 'transiiditsoonist', 'lemma': 'transiiditsoon', 'root': 'transiidi_tsoon', 'root_tokens': ['transiidi', 'tsoon'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'S'}]),
Span('väljumata', [{'normalized_text': 'väljumata', 'lemma': 'väljuma', 'root': 'välju', 'root_tokens': ['välju'], 'ending': 'mata', 'clitic': '', 'form': 'mata', 'partofspeech': 'V'}]),
Span('edasi', [{'normalized_text': 'edasi', 'lemma': 'edasi', 'root': 'edasi', 'root_tokens': ['edasi'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('Soome', [{'normalized_text': 'Soome', 'lemma': 'Soome', 'root': 'Soome', 'root_tokens': ['Soome'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('lendama', [{'normalized_text': 'lendama', 'lemma': 'lendama', 'root': 'lenda', 'root_tokens': ['lenda'], 'ending': 'ma', 'clitic': '', 'form': 'ma', 'partofspeech': 'V'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [6]:
Enc2017Preprocessor.create_df_enc2017(
    jsons=jsons,
    output_filename=ENC2017_DIRS["processed"] / "enc2017_morph_analysis.parquet",
)

Beginning to create dataset from JSON files.


100%|██████████| 18486/18486 [17:27<00:00, 17.65it/s]  


Morphological tagging completed successfully
Tagged texts saved to ..\data\enc2017\processed\enc2017_morph_analysis.parquet



<a id='analysis_of_the_results_1'></a>


### Analysis of the results


In [7]:
pq_enc = pd.read_parquet(ENC2017_DIRS["processed"] / "enc2017_morph_analysis.parquet")

In [8]:
display(pq_enc.head())

,sentence_id,words,form,pos,labels,type,source
0,0,Teisipäeval,sg ad,S,sg ad_S,periodicals,nc_10114_622631.json
1,0,saabus,s,V,s_V,periodicals,nc_10114_622631.json
2,0,Iisraelist,sg el,H,sg el_H,periodicals,nc_10114_622631.json
3,0,Tallinna,sg g,H,sg g_H,periodicals,nc_10114_622631.json
4,0,lennujaama,sg g,S,sg g_S,periodicals,nc_10114_622631.json


In [9]:
pq_enc.size

72731533

In [10]:
# Table of pos counts with empty forms ("")
display(pq_enc.groupby("pos")["form"].apply(lambda x: (x == "").sum()))

pos
A      78712
C          0
D     905164
G      19511
H        377
I      10978
J     643654
K     204167
N          0
O          0
P         13
S          0
U          0
V         28
X       2151
Y          0
Z    1830767
Name: form, dtype: int64

In [11]:
# Examples of empty forms ("") with POS tag "A"
display(pq_enc[(pq_enc["pos"] == "A") & (pq_enc["form"] == "")])

,sentence_id,words,form,pos,labels,type,source
22,1,broneeritud,,A,A,periodicals,nc_10114_622631.json
27,1,ostetud,,A,A,periodicals,nc_10114_622631.json
157,10,möödunud,,A,A,periodicals,nc_10114_622633.json
214,12,kutsunud,,A,A,periodicals,nc_10114_622633.json
306,18,majajäetud,,A,A,periodicals,nc_10114_622635.json
...,...,...,...,...,...,...,...
10389671,736214,ettevalmistamata,,A,A,wikipedia,wiki17_99902_x.json
10389741,736217,ebaõnnestunud,,A,A,wikipedia,wiki17_99902_x.json
10389990,736224,glasuurimata,,A,A,wikipedia,wiki17_99913_x.json
10390014,736226,Põletatud,,A,A,wikipedia,wiki17_99913_x.json


In [12]:
text = estnltk.converters.json_to_text(file=jsons[0])

if "morph_analysis" in text.layers:
    display(text.sentences[1].morph_analysis)

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Selleks', [{'normalized_text': 'Selleks', 'lemma': 'see', 'root': 'see', 'root_tokens': ['see'], 'ending': 'ks', 'clitic': '', 'form': 'sg tr', 'partofspeech': 'P'}]),
Span('oli', [{'normalized_text': 'oli', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('tal', [{'normalized_text': 'tal', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'P'}]),
Span('lennukis', [{'normalized_text': 'lennukis', 'lemma': 'lennuk', 'root': 'lennuk', 'root_tokens': ['lennuk'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('koht', [{'normalized_text': 'koht', 'lemma': 'koht', 'root': 'koht', 'root_tokens': ['koht'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('broneeritud', [{'normalized_text': 'broneeritud', 'lemma': 'broneeritud', 'root': 'broneeri=tud', 'root_tokens': ['broneeritud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}, {'normalized_text': 'broneeritud', 'lemma': 'broneeritud', 'root': 'broneeri=tud', 'root_tokens': ['broneeritud'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}, {'normalized_text': 'broneeritud', 'lemma': 'broneerima', 'root': 'broneeri', 'root_tokens': ['broneeri'], 'ending': 'tud', 'clitic': '', 'form': 'tud', 'partofspeech': 'V'}, {'normalized_text': 'broneeritud', 'lemma': 'broneeritud', 'root': 'broneeri=tud', 'root_tokens': ['broneeritud'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'A'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('kuid', [{'normalized_text': 'kuid', 'lemma': 'kuid', 'root': 'kuid', 'root_tokens': ['kuid'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('piletit', [{'normalized_text': 'piletit', 'lemma': 'pilet', 'root': 'pilet', 'root_tokens': ['pilet'], 'ending': 't', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('veel', [{'normalized_text': 'veel', 'lemma': 'veel', 'root': 'veel', 'root_tokens': ['veel'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('ostetud', [{'normalized_text': 'ostetud', 'lemma': 'ostetud', 'root': 'oste=tud', 'root_tokens': ['ostetud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}, {'normalized_text': 'ostetud', 'lemma': 'ostma', 'root': 'ost', 'root_tokens': ['ost'], 'ending': 'tud', 'clitic': '', 'form': 'tud', 'partofspeech': 'V'}, {'normalized_text': 'ostetud', 'lemma': 'ostetud', 'root': 'oste=tud', 'root_tokens': ['ostetud'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}, {'normalized_text': 'ostetud', 'lemma': 'ostetud', 'root': 'oste=tud', 'root_tokens': ['ostetud'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'A'}]),
Span('polnud', [{'normalized_text': 'polnud', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'nud', 'clitic': '', 'form': 'neg nud', 'partofspeech': 'V'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [ ]:
# Read in unique labels from the unique_labels_old.json file
with open(pathlib.Path("../outputs/") / "unique_labels_old.json", "r") as f:
    unique_labels = json.load(f)

print(f"Number of unique labels: {len(unique_labels)}")

Number of unique labels: 1563


In [25]:
# For each sentence_id in the dataset pq_enc, check if all the labels in the sentence are present in the unique_labels list
# Count all the labels that are not in the unique_labels list
from collections import Counter

counter = Counter()
value_counts_pq_enc = pq_enc["labels"].value_counts()
for value in value_counts_pq_enc.index:
    if value not in unique_labels:
        counter[value] += value_counts_pq_enc[value]
print(
    f"Number of labels in dataset that are not in unique_labels: {sum(counter.values())}"
)
print(f"Labels in dataset that are not in unique_labels: {counter.items()}")

Number of labels in dataset that are not in unique_labels: 4
Labels in dataset that are not in unique_labels: dict_items([('neg da_V', np.int64(4))])


In [26]:
# Group by sentence_id and source and check if any of the labels in the sentence contain unknown labels
source_sentence_id_groups = pq_enc.groupby(["sentence_id", "source"])
for (sentence_id, source), group in source_sentence_id_groups:
    if any(group["labels"].str.contains("neg da_V")):
        print(f"Sentence ID: {sentence_id}, Source: {source}")
        print(group[["words", "labels"]])
        display(
            pq_enc[
                (pq_enc["sentence_id"] == sentence_id) & (pq_enc["source"] == source)
            ]
        )
        break

Sentence ID: 153208, Source: nc_11841_733481.json
                  words    labels
2008856            Ning         J
2008857             eks         D
2008858         furoori     adt_S
2008859             ole       o_V
2008860       tekitanud     nud_V
2008861              ka         D
2008862             see    sg n_P
2008863               ,         Z
2008864              et         J
2008865           oleme      me_V
2008866         lubanud     nud_V
2008867             oma    sg g_P
2008868           rahva    sg g_S
2008869     geenivaramu    sg n_S
2008870            luua      da_V
2008871               ,         Z
2008872             kus         D
2008873             iga    sg n_P
2008874           Matsi    sg g_H
2008875              ja         J
2008876            Mari    sg n_S
2008877               ,         Z
2008878             aga         J
2008879              ka         D
2008880           Ivani    sg g_H
2008881          geenid    pl n_S
2008882               (         

,sentence_id,words,form,pos,labels,type,source
2008856,153208,Ning,,J,J,fiction,nc_11841_733481.json
2008857,153208,eks,,D,D,fiction,nc_11841_733481.json
2008858,153208,furoori,adt,S,adt_S,fiction,nc_11841_733481.json
2008859,153208,ole,o,V,o_V,fiction,nc_11841_733481.json
2008860,153208,tekitanud,nud,V,nud_V,fiction,nc_11841_733481.json
2008861,153208,ka,,D,D,fiction,nc_11841_733481.json
2008862,153208,see,sg n,P,sg n_P,fiction,nc_11841_733481.json
2008863,153208,",",,Z,Z,fiction,nc_11841_733481.json
2008864,153208,et,,J,J,fiction,nc_11841_733481.json
2008865,153208,oleme,me,V,me_V,fiction,nc_11841_733481.json


In [29]:
# Replace the "neg da" form and "neg da_V" label in the pq_enc DataFrame with "da" and "da_V", respectively
# Since these labels are not defined in tables of morphological categories, replace them with "da" and "da_V", which are defined in the tables and are the most similar to the original labels.
pq_enc["form"] = pq_enc["form"].replace("neg da", "da")
pq_enc["labels"] = pq_enc["labels"].replace("neg da_V", "da_V")

In [30]:
# For each sentence_id in the dataset pq_enc, check if all the labels in the sentence are present in the unique_labels list
# Count all the labels that are not in the unique_labels list
counter = Counter()
value_counts_pq_enc = pq_enc["labels"].value_counts()
for value in value_counts_pq_enc.index:
    if value not in unique_labels:
        counter[value] += value_counts_pq_enc[value]
print(
    f"Number of labels in dataset that are not in unique_labels: {sum(counter.values())}"
)
print(f"Labels in dataset that are not in unique_labels: {counter.items()}")

Number of labels in dataset that are not in unique_labels: 0
Labels in dataset that are not in unique_labels: dict_items([])


In [31]:
# Save the updated pq_enc DataFrame to a new parquet file
pq_enc.to_parquet(ENC2017_DIRS["processed"] / "enc2017_morph_analysis_updated.parquet")

<a id='further_preprocessing_1'></a>


### Further preprocessing


In [32]:
# Read in the data
data = pd.read_parquet(
    ENC2017_DIRS["processed"] / "enc2017_morph_analysis_updated.parquet"
)
display(len(data))
display(data.head())

10390219

,sentence_id,words,form,pos,labels,type,source
0,0,Teisipäeval,sg ad,S,sg ad_S,periodicals,nc_10114_622631.json
1,0,saabus,s,V,s_V,periodicals,nc_10114_622631.json
2,0,Iisraelist,sg el,H,sg el_H,periodicals,nc_10114_622631.json
3,0,Tallinna,sg g,H,sg g_H,periodicals,nc_10114_622631.json
4,0,lennujaama,sg g,S,sg g_S,periodicals,nc_10114_622631.json


In [33]:
print(data.head())

   sentence_id        words   form pos   labels         type  \
0            0  Teisipäeval  sg ad   S  sg ad_S  periodicals   
1            0       saabus      s   V      s_V  periodicals   
2            0   Iisraelist  sg el   H  sg el_H  periodicals   
3            0     Tallinna   sg g   H   sg g_H  periodicals   
4            0   lennujaama   sg g   S   sg g_S  periodicals   

                 source  
0  nc_10114_622631.json  
1  nc_10114_622631.json  
2  nc_10114_622631.json  
3  nc_10114_622631.json  
4  nc_10114_622631.json  


In [34]:
shuffled_df = Enc2017Preprocessor.shuffle_blocks_by_source(
    data, source_col="source", seed=seed
)

In [35]:
sampled_df = Enc2017Preprocessor.stratified_sample_by_type(
    shuffled_df,
    N=1000000,
    source_col="source",
    type_col="type",
    seed=seed,
    verbose=True,
)

Type 'wikipedia': quota=200000, selected=200001, overshoot=1
Type 'periodicals': quota=200000, selected=200000, overshoot=0
Type 'blogs_and_forums': quota=200000, selected=200000, overshoot=0
Type 'science': quota=200000, selected=200240, overshoot=240
Type 'fiction': quota=200000, selected=200324, overshoot=324


In [36]:
display(len(sampled_df))
display(sampled_df.head())

1000565

,sentence_id,words,form,pos,labels,type,source
0,705070,"""",,Z,Z,wikipedia,wiki17_56226_x.json
1,705070,Eetika,sg g,S,sg g_S,wikipedia,wiki17_56226_x.json
2,705070,ja,,J,J,wikipedia,wiki17_56226_x.json
3,705070,filosoofia,sg g,S,sg g_S,wikipedia,wiki17_56226_x.json
4,705070,piirid,pl n,S,pl n_S,wikipedia,wiki17_56226_x.json


In [ ]:
sampled_df.to_parquet(ENC2017_DIRS["processed"] / "sampled_data.parquet", index=False)

In [39]:
from sklearn.model_selection import train_test_split

# Split whole dataset (sampled_df) into train and testset
# Maintain sentence integrity by splitting on sentence level
sentences = sampled_df["sentence_id"].unique()
train_sentences, test_sentences = train_test_split(
    sentences, test_size=0.25, random_state=seed
)
train_data = sampled_df[sampled_df["sentence_id"].isin(train_sentences)]
test_data = sampled_df[sampled_df["sentence_id"].isin(test_sentences)]

In [41]:
# Save the train and test sets to CSV files
train_data.to_parquet(
    ENC2017_DIRS["processed"] / "sampled_train.parquet",
    index=False,
)
test_data.to_parquet(
    ENC2017_DIRS["processed"] / "sampled_test.parquet",
    index=False,
)

<a id='estonian_ud_edt_corpus'></a>


## Estonian Universal Dependencies' EDT [corpus](https://github.com/UniversalDependencies/UD_Estonian-EDT)


<a id='extracting_the_data_2'></a>


### Extracting the data


In [4]:
import numpy as np
import pandas as pd
import estnltk
import tqdm

from scripts.preprocessing.ud_et_edt.extract import Extractor as UdEtEdtExtractor
from scripts.preprocessing.ud_et_edt.preprocess import (
    Preprocessor as UdEtEdtPreprocessor,
)

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [43]:
ud_corpus_dir = UD_ET_EDT_DIRS["raw"] / "UD_Estonian-EDT-r2.17"  # UD Corpus location
output_dir = UD_ET_EDT_DIRS["processed"] / "UD_converted"  # Output directory

In [4]:
UdEtEdtExtractor.convert_ud_to_vabamorf(ud_corpus_dir, output_dir, skip_existing=True)

Converting UD corpus to Vabamorf format: 100%|██████████| 3/3 [00:25<00:00,  8.61s/file]
Converting UD morphosyntactic annotations to Vabamorf-like annotations and writing to JSON: 100%|██████████| 43/43 [00:25<00:00,  1.66text/s]


<a id='gathering_the_data_2'></a>


### Gathering the data


In [44]:
jsons_dir = UD_ET_EDT_DIRS["processed"] / "UD_converted"
jsons = list(jsons_dir.glob("*.json"))

In [45]:
UdEtEdtPreprocessor.create_df_ud_corpus(
    jsons=jsons,
    tokenizer="ud_morph_reduced",
    output_filename=UD_ET_EDT_DIRS["processed"] / "ud_edt_morph_analysis.parquet",
)

Beginning to morphologically tag file by file. This can take a while.


Processing files: 100%|██████████| 43/43 [00:49<00:00,  1.15s/file]

Morphological tagging completed successfully
Tagged texts saved to ..\data\ud_et_edt\processed\ud_edt_morph_analysis.parquet



<a id='analysis_of_the_results_2'></a>


### Analysis of the results


In [143]:
pq_ud_et_edt = pd.read_parquet(
    UD_ET_EDT_DIRS["processed"] / "ud_edt_morph_analysis.parquet"
)

In [144]:
display(pq_ud_et_edt.head())
print(
    "Number of sentences in UD_ET_EDT dataset:", pq_ud_et_edt["sentence_id"].nunique()
)

,sentence_id,words,form,pos,labels,file_prefix,source
0,0,Aga,,J,J,aja_ee199920,et_edt-ud-dev_000.json
1,0,mulle,sg all,P,sg all_P,aja_ee199920,et_edt-ud-dev_000.json
2,0,tundub,b,V,b_V,aja_ee199920,et_edt-ud-dev_000.json
3,0,",",,Z,Z,aja_ee199920,et_edt-ud-dev_000.json
4,0,et,,J,J,aja_ee199920,et_edt-ud-dev_000.json


Number of sentences in UD_ET_EDT dataset: 30930


In [145]:
# Table of pos counts with empty forms ("")
display(pq_ud_et_edt.groupby("pos")["form"].apply(lambda x: (x == "").sum()))

pos
''           1
2            0
A         4357
C            1
D        41626
DET          0
DT           1
G          644
H           33
I          319
J        24845
K         9249
N         5890
NN           1
O         1534
P            8
PRP          0
RB           1
S          130
T         1085
U            4
V          194
VBZ          1
X          399
Y         3410
Z        71687
``           1
p            0
place        0
Name: form, dtype: int64

In [146]:
# Examples of empty forms ("") with POS tag "A"
display(
    pq_ud_et_edt[(pq_ud_et_edt["pos"] == "A") & (pq_ud_et_edt["form"] == "")].head()
)

,sentence_id,words,form,pos,labels,file_prefix,source
5,0,kogu,,A,A,aja_ee199920,et_edt-ud-dev_000.json
618,37,veendunud,,A,A,aja_ee199920,et_edt-ud-dev_000.json
769,46,Möödunud,,A,A,aja_ee199920,et_edt-ud-dev_000.json
873,54,alasti,,A,A,aja_ee199920,et_edt-ud-dev_000.json
1187,77,kureeritud,,A,A,aja_ee199920,et_edt-ud-dev_000.json


In [147]:
# Examples of empty forms ("") with POS tag "N"
display(
    pq_ud_et_edt[(pq_ud_et_edt["pos"] == "N") & (pq_ud_et_edt["form"] == "")].head()
)

,sentence_id,words,form,pos,labels,file_prefix,source
258,14,1994,,N,N,aja_ee199920,et_edt-ud-dev_000.json
372,20,18,,N,N,aja_ee199920,et_edt-ud-dev_000.json
684,41,8,,N,N,aja_ee199920,et_edt-ud-dev_000.json
685,41,½,,N,N,aja_ee199920,et_edt-ud-dev_000.json
741,45,8,,N,N,aja_ee199920,et_edt-ud-dev_000.json


In [148]:
text = estnltk.converters.json_to_text(file=jsons[0])

display(text.sentences[0].ud_morph_reduced)

Layer(name='ud_morph_reduced', attributes=('lemma', 'pos', 'form'), spans=SL[Span('Aga', [{'lemma': 'aga', 'pos': 'J', 'form': ''}]),
Span('mulle', [{'lemma': 'mina', 'pos': 'P', 'form': 'sg all'}]),
Span('tundub', [{'lemma': 'tundu', 'pos': 'V', 'form': 'b'}]),
Span(',', [{'lemma': ',', 'pos': 'Z', 'form': ''}]),
Span('et', [{'lemma': 'et', 'pos': 'J', 'form': ''}]),
Span('kogu', [{'lemma': 'kogu', 'pos': 'A', 'form': ''}]),
Span('maailm', [{'lemma': 'maa_ilm', 'pos': 'S', 'form': 'sg n'}]),
Span('ootab', [{'lemma': 'oota', 'pos': 'V', 'form': 'b'}]),
Span('muusikamaailmalt', [{'lemma': 'muusika_maa_ilm', 'pos': 'S', 'form': 'sg abl'}]),
Span('midagi', [{'lemma': 'miski', 'pos': 'P', 'form': 'sg p'}]),
Span('erutavalt', [{'lemma': 'erutavalt', 'pos': 'D', 'form': ''}]),
Span('uut', [{'lemma': 'uus', 'pos': 'A', 'form': 'sg p'}]),
Span('minimalismi', [{'lemma': 'minimalism', 'pos': 'S', 'form': 'sg g'}]),
Span('kõrvale', [{'lemma': 'kõrvale', 'pos': 'K', 'form': ''}]),
Span('.', [{'lemma': '.', 'pos': 'Z', 'form': ''}])])

In [149]:
# Read in unique labels from the unique_labels_old.json file
with open(pathlib.Path("../outputs/") / "unique_labels_old.json", "r") as f:
    unique_labels = json.load(f)

print(f"Number of unique labels: {len(unique_labels)}")

Number of unique labels: 1563


In [150]:
# For each sentence_id in the dataset pq_ud_et_edt, check if all the labels in the sentence are present in the unique_labels list
# Count all the labels that are not in the unique_labels list
from collections import Counter

counter = Counter()
value_counts_pq_ud_et_edt = pq_ud_et_edt["labels"].value_counts()
for value in value_counts_pq_ud_et_edt.index:
    if value not in unique_labels:
        counter[value] += value_counts_pq_ud_et_edt[value]
print(
    f"Number of labels in dataset that are not in unique_labels: {sum(counter.values())}"
)
print(f"Labels in dataset that are not in unique_labels: {counter.items()}")

Number of labels in dataset that are not in unique_labels: 1137
Labels in dataset that are not in unique_labels: dict_items([('T', np.int64(1085)), ('sg n_T', np.int64(20)), ('sg g_T', np.int64(5)), ('pl n_T', np.int64(3)), ('sg n_D', np.int64(3)), ('sg p_T', np.int64(3)), ('sg g_DET', np.int64(1)), ('sg p_D', np.int64(1)), ('DT', np.int64(1)), ("''", np.int64(1)), ('NN', np.int64(1)), ('sg ad_D', np.int64(1)), ('sg n_p', np.int64(1)), ('sg g_place', np.int64(1)), ('sg in_T', np.int64(1)), ('sg ad_T', np.int64(1)), ('sg all_T', np.int64(1)), ('sg abl_T', np.int64(1)), ('VBZ', np.int64(1)), ('``', np.int64(1)), ('RB', np.int64(1)), ('sg el_T', np.int64(1)), ('sg n_PRP', np.int64(1)), ('sg g_2', np.int64(1))])


In [151]:
# Group by sentence_id and source and check if any of the labels in the sentence contain unknown labels
source_sentence_id_groups = pq_ud_et_edt.groupby(["sentence_id", "source"])
i = 0
for (sentence_id, source), group in source_sentence_id_groups:
    if any(group["labels"].str.contains("sg n_D")):
        print(f"Sentence ID: {sentence_id}, Source: {source}")
        print(group[["words", "labels"]])
        i += 1
    if i >= 10:
        break

Sentence ID: 11124, Source: et_edt-ud-train_017.json
                 words     labels
156163            Äsja          D
156164        lõppenud          A
156165         Euroopa     sg g_H
156166           Liidu     sg g_H
156167        ülemkogu     sg g_S
156168    kokkusaamine     sg n_S
156169               ,          Z
156170             kus          D
156171        kohtusid      sid_V
156172            EL-i     sg g_Y
156173  liikmesriikide     pl g_S
156174           juhid     pl n_S
156175               ,          Z
156176              ei      neg_V
156177           olnud  neg nud_V
156178            sama     sg n_D
156179  vaatemänguline     sg n_A
156180             kui          J
156181          mõnigi     sg n_P
156182         varasem     sg n_C
156183   tippkohtumine     sg n_S
156184               ,          Z
156185           mille     sg g_P
156186           puhul          K
156187    liikmesmaade     pl g_S
156188           huvid     pl n_S
156189        teravalt       

In [152]:
# Replace "sg g_2", "sg g_place" with "sg g_S"
# Replace "sg g_DET" with "sg g_P"
# Replace "sg n_PRP", "VBZ", "RB", "DT", "NN", "''", "``" with "-"
# Replace "sg n_p" with "sg n_P"
# Replace "sg ad_D" with "sg ad_S"
# Replace "sg p_D" with "D"
# Replace "sg n_D" with "D"

pq_ud_et_edt["labels"] = pq_ud_et_edt["labels"].replace(
    {
        "sg g_2": "sg g_S",
        "sg g_place": "sg g_S",
        "sg g_DET": "sg g_P",
        "sg n_PRP": "-",
        "VBZ": "-",
        "RB": "-",
        "DT": "-",
        "NN": "-",
        "''": "-",
        "``": "-",
        "sg n_p": "sg n_P",
        "sg ad_D": "sg ad_S",
        "sg p_D": "D",
        "sg n_D": "D",
    }
)

pq_ud_et_edt["pos"] = pq_ud_et_edt["pos"].replace(
    {
        "2": "S",
        "place": "S",
        "DET": "P",
        "PRP": "-",
        "VBZ": "-",
        "RB": "-",
        "DT": "-",
        "NN": "-",
        "''": "-",
        "``": "-",
        "p": "sg n_P",
    }
)

# For these find the word, sentence_id and source of the occurrence of these labels and replace the form and pos labels in the pq_ud_et_edt DataFrame with the new labels. The labels to be replaced are:
# "sg ad_D": "sg ad_S", <- POS change
# "sg p_D": "D", <- form change
# "sg n_D": "D", <- form change
source_sentence_id_groups = pq_ud_et_edt.groupby(["sentence_id", "source"])
for (sentence_id, source), group in source_sentence_id_groups:
    if any(group["labels"].str.contains("sg ad_D")):
        print(f"Sentence ID: {sentence_id}, Source: {source}")
        # Replace POS tag "D" with "S" for the current sentence_id and source
        pq_ud_et_edt.loc[
            (pq_ud_et_edt["sentence_id"] == sentence_id)
            & (pq_ud_et_edt["source"] == source)
            & (pq_ud_et_edt["labels"].str.contains("sg ad_D")),
            "pos",
        ] = "S"
    elif any(group["labels"].str.contains("sg p_D")):
        print(f"Sentence ID: {sentence_id}, Source: {source}")
        # Replace form label "sg p" with "" for the current sentence_id and source
        pq_ud_et_edt.loc[
            (pq_ud_et_edt["sentence_id"] == sentence_id)
            & (pq_ud_et_edt["source"] == source)
            & (pq_ud_et_edt["labels"].str.contains("sg p_D")),
            "form",
        ] = ""
    elif any(group["labels"].str.contains("sg n_D")):
        print(f"Sentence ID: {sentence_id}, Source: {source}")
        # Replace form label "sg n" with "" for the current sentence_id and source
        pq_ud_et_edt.loc[
            (pq_ud_et_edt["sentence_id"] == sentence_id)
            & (pq_ud_et_edt["source"] == source)
            & (pq_ud_et_edt["labels"].str.contains("sg n_D")),
            "form",
        ] = ""
    # Remove sentences where "T" (matched case) is present in the labels, as these are not defined in the tables of morphological categories and so not used in training
    elif any(group["labels"].str.contains("T")):
        # print(f"Sentence ID: {sentence_id}, Source: {source}")
        # Rename the form to REMOVE ME for all words in the current sentence_id and source where "T" is present in the labels, so that these sentences can be removed from the dataset in the next step
        pq_ud_et_edt.loc[
            (pq_ud_et_edt["sentence_id"] == sentence_id)
            & (pq_ud_et_edt["source"] == source),
            "form",
        ] = "REMOVE ME"
    elif any(group["labels"].str.contains("-")):
        # Rename the form to REMOVE ME for all words in the current sentence_id and source where "-" is present in the labels, so that these sentences can be removed from the dataset in the next step
        pq_ud_et_edt.loc[
            (pq_ud_et_edt["sentence_id"] == sentence_id)
            & (pq_ud_et_edt["source"] == source),
            "form",
        ] = "REMOVE ME"
    else:
        continue

# Remove sentences where the form is "REMOVE ME"
pq_ud_et_edt = pq_ud_et_edt[pq_ud_et_edt["form"] != "REMOVE ME"]

In [153]:
# For each sentence_id in the dataset pq_ud_et_edt, check if all the labels in the sentence are present in the unique_labels list
# Count all the labels that are not in the unique_labels list
from collections import Counter

counter = Counter()
value_counts_pq_ud_et_edt = pq_ud_et_edt["labels"].value_counts()
for value in value_counts_pq_ud_et_edt.index:
    if value not in unique_labels:
        counter[value] += value_counts_pq_ud_et_edt[value]
print(
    f"Number of labels in dataset that are not in unique_labels: {sum(counter.values())}"
)
print(f"Labels in dataset that are not in unique_labels: {counter.items()}")

Number of labels in dataset that are not in unique_labels: 0
Labels in dataset that are not in unique_labels: dict_items([])


In [154]:
display(pq_ud_et_edt.head())
print(
    "Number of sentences in UD_ET_EDT dataset:", pq_ud_et_edt["sentence_id"].nunique()
)

,sentence_id,words,form,pos,labels,file_prefix,source
0,0,Aga,,J,J,aja_ee199920,et_edt-ud-dev_000.json
1,0,mulle,sg all,P,sg all_P,aja_ee199920,et_edt-ud-dev_000.json
2,0,tundub,b,V,b_V,aja_ee199920,et_edt-ud-dev_000.json
3,0,",",,Z,Z,aja_ee199920,et_edt-ud-dev_000.json
4,0,et,,J,J,aja_ee199920,et_edt-ud-dev_000.json


Number of sentences in UD_ET_EDT dataset: 30515


In [155]:
# Save the updated pq_ud_et_edt DataFrame to a new parquet file
pq_ud_et_edt.to_parquet(
    UD_ET_EDT_DIRS["processed"] / "ud_edt_morph_analysis_updated.parquet"
)

<a id='further_preprocessing_2'></a>


### Further preprocessing


In [5]:
# Split whole dataset (pq_ud_et_edt) into train, dev and test sets using the source column filenames
# Train has "train" in the source filename, dev has "dev" in the source filename and test has "test" in the source filename
pq_ud_ed_edt = pd.read_parquet(
    UD_ET_EDT_DIRS["processed"] / "ud_edt_morph_analysis_updated.parquet"
)

train_data = pq_ud_ed_edt[pq_ud_ed_edt["source"].str.contains("train")]
dev_data = pq_ud_ed_edt[pq_ud_ed_edt["source"].str.contains("dev")]
test_data = pq_ud_ed_edt[pq_ud_ed_edt["source"].str.contains("test")]
print(f"Rows in overall dataset: {len(pq_ud_ed_edt)}")
print(
    f"  Rows in train set: {len(train_data)} ({len(train_data) / len(pq_ud_ed_edt) * 100:.2f}%)"
)
print(
    f"  Rows in dev set: {len(dev_data)} ({len(dev_data) / len(pq_ud_ed_edt) * 100:.2f}%)"
)
print(
    f"  Rows in test set: {len(test_data)} ({len(test_data) / len(pq_ud_ed_edt) * 100:.2f}%)"
)

Rows in overall dataset: 428674
  Rows in train set: 336868 (78.58%)
  Rows in dev set: 43997 (10.26%)
  Rows in test set: 47809 (11.15%)


In [157]:
# Save the train, dev and test sets to new parquet files
train_data.to_parquet(
    UD_ET_EDT_DIRS["processed"] / "ud_edt_train.parquet",
    index=False,
)
dev_data.to_parquet(
    UD_ET_EDT_DIRS["processed"] / "ud_edt_dev.parquet",
    index=False,
)
test_data.to_parquet(
    UD_ET_EDT_DIRS["processed"] / "ud_edt_test.parquet",
    index=False,
)

<a id='homonym_disambiguation_corpus'></a>


## Homonym disambiguation corpus


In [ ]:
import pandas as pd
import estnltk, estnltk.converters, estnltk.taggers

from scripts.preprocessing.homonym.extract import Extractor as HomonymExtractor
from scripts.preprocessing.homonym.preprocess import Preprocessor as HomonymPreprocessor

<a id='jsonl_file_3'></a>


### JSONL file


<a id='gathering_the_data_32'></a>


#### Gathering the data


In [ ]:
import os

output_dir = str(HOMONYMS_DIRS["annotations"] / "random_pick_jl")
input_forms_file = str(HOMONYMS_DIRS["annotations"] / "homonymous_forms_1_16_17_19.csv")
input_sent_file = str(
    HOMONYMS_DIRS["annotations"] / "all_enc_koond_homonymous_forms_sentences.jl"
)
print(
    os.path.exists(output_dir),
    os.path.exists(input_forms_file),
    os.path.exists(input_sent_file),
)

True True True


In [ ]:
%run "../scripts/preprocessing/homonym/03_pick_1000_sentences_from_each_infl_type.py" --pick-target 2000 --all-sentences-uniq --discard-previously-picked --output-dir "../data/homonymous_word_forms/annotations/random_pick_jl/" --input-forms-file "../data/homonymous_word_forms/annotations/homonymous_forms_1_16_17_19.csv" --input-sent-file "../data/homonymous_word_forms/annotations/all_enc_koond_homonymous_forms_sentences.jl"

Total homonymous forms:  13549

     homonymous forms   
    with lemma == form:  11827

   INFLECTION TYPE 1:  homonymous word forms :  3012
   INFLECTION TYPE 16:  homonymous word forms :  2206
   INFLECTION TYPE 19:  homonymous word forms :  565
   INFLECTION TYPE 17:  homonymous word forms :  439



0it [00:00, ?it/s]

4883131it [00:34, 142160.55it/s]


 Loaded 8000 previously picked sentences.
 Removing previously picked sentences from all available sentences.

    INFLECTION TYPE 16: number of homonymous form sentences:  2952882
                           uniq sentences:  2320196 / 2952882 (78.57%)
                propernoun word sentences:  2419386 / 2952882 (81.93%)
all uniq words coverage (from VM lexicon):  2043 / 2206 (92.61%)
    INFLECTION TYPE 1: number of homonymous form sentences:  2428205
                           uniq sentences:  2042663 / 2428205 (84.12%)
                propernoun word sentences:  895064 / 2428205 (36.86%)
all uniq words coverage (from VM lexicon):  2631 / 3012 (87.35%)
    INFLECTION TYPE 17: number of homonymous form sentences:  1427173
                           uniq sentences:  1301863 / 1427173 (91.22%)
                propernoun word sentences:  2539 / 1427173 (0.18%)
all uniq words coverage (from VM lexicon):  427 / 439 (97.27%)
    INFLECTION TYPE 19: number of homonymous form sentences:  2784

In [11]:
# Convert randomly picked sentences from JSONL to LabelStudio JSON format
%run "../scripts/preprocessing/homonym/04_export_to_labelstudio.py" --input-dir "../data/homonymous_word_forms/annotations/random_pick_jl_2/" --output-dir "../data/homonymous_word_forms/annotations/"

<a id='labelstudio_json_files_3'></a>


### LabelStudio JSON files


<a id='extracting_the_data_31'></a>


#### Extracting the data


In [ ]:
input_dir = str(HOMONYMS_DIRS["annotations"])
input_files = HomonymExtractor.extract(input_dir=input_dir)

Found 8 input files.


<a id='gathering_the_data_31'></a>


#### Gathering the data


In [6]:
HomonymPreprocessor.create_sentences_df(
    input_files=input_files,
    output_dir=str(HOMONYMS_DIRS["processed"]),
    do_individual_dfs=False,
    do_overall_df=True,
)

Processing file: ..\data\homonymous_word_forms\annotations\1\infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json (inflection type 1)
Processing file: ..\data\homonymous_word_forms\annotations\1\infl_type_16_1000_v1_project-2-at-2024-11-21-19-27-9e8ae0c2.json (inflection type 16)
Processing file: ..\data\homonymous_word_forms\annotations\1\infl_type_17_1000_v1.json (inflection type 17)
Processing file: ..\data\homonymous_word_forms\annotations\1\infl_type_19_1000_v1_project-6-at-2025-11-15-14-13-42753676.json (inflection type 19)
Processing file: ..\data\homonymous_word_forms\annotations\2\infl_type_01_1000_v2_project-5-at-2024-12-11-21-53-280753b4.json (inflection type 1)
Processing file: ..\data\homonymous_word_forms\annotations\2\infl_type_16_1000_v2_project-5-at-2025-01-03-01-42-87bca577.json (inflection type 16)
Processing file: ..\data\homonymous_word_forms\annotations\2\infl_type_17_1000_v2.json (inflection type 17)
Processing file: ..\data\homonymous_word_forms\anno

In [35]:
HomonymPreprocessor.create_model_df(
    input_files=input_files,
    output_dir=str(HOMONYMS_DIRS["processed"]),
    do_individual_dfs=False,
    do_overall_df=True,
)

Saved overall processed data to ..\data\homonymous_word_forms\processed\homonyms_overall.parquet


<a id='analysis_of_the_results_31'></a>


#### Analysis of the results


In [7]:
homonym_dataset = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall.parquet"
)
homonym_sentences_dataset = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_sentences.parquet"
)

In [8]:
print(homonym_dataset.head())
print(homonym_sentences_dataset.head())

   sentence_id       words form pos labels  infl_type  \
0            0  Edinburghi    -   -      -          1   
1            0     agulite    -   -      -          1   
2            0        mehe    -   -      -          1   
3            0      Irvine    -   -      -          1   
4            0      Welshi    -   -      -          1   

                                                             source  
0  infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json  
1  infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json  
2  infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json  
3  infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json  
4  infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json  
   num  inflection_type  \
0    1                1   
1    1                1   
2    1                1   
3    1                1   
4    1                1   

                                                                          

In [9]:
display(homonym_dataset.head())
display(homonym_sentences_dataset.head())

,sentence_id,words,form,pos,labels,infl_type,source
0,0,Edinburghi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,0,agulite,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,0,mehe,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,0,Irvine,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,0,Welshi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


,num,inflection_type,sentence,word,word_span,label,source
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [10]:
display(homonym_dataset["form"].value_counts())
display(homonym_sentences_dataset["label"].value_counts())

form
-       172100
sg g      4457
sg n      2445
sg p       890
adt         94
Name: count, dtype: int64

label
[sg g]    4457
[sg n]    2445
[sg p]     890
[adt]       94
Name: count, dtype: int64

In [11]:
print(homonym_dataset["labels"].value_counts())

labels
-         172100
sg g_S      2394
sg g_H      1945
sg n_S      1235
sg n_H       844
sg p_S       831
sg n_A       210
sg n_P       121
sg g_A        67
adt_S         66
sg n_N        35
sg g_P        35
adt_H         28
sg p_H        26
sg p_P        25
sg g_N        15
sg p_A         8
sg g_V         1
Name: count, dtype: int64


In [12]:
# Print the words list of the sentence that contains a word with the label "sg g_V"
sentence_with_label = homonym_dataset[homonym_dataset["labels"] == "sg g_V"][
    "sentence_id"
].iloc[0]
words_in_sentence = homonym_dataset[
    homonym_dataset["sentence_id"] == sentence_with_label
]["words"].tolist()
print(words_in_sentence)

# Replace the label "sg g_V" with "sg g_H" in the homonym_dataset
homonym_dataset["labels"] = homonym_dataset["labels"].replace("sg g_V", "sg g_H")

# Do the same replacement in the homonym_sentences_dataset
homonym_sentences_dataset["label"] = homonym_sentences_dataset["label"].replace(
    "sg g_V", "sg g_H"
)

['"', 'Kui', 'veel', 'aasta-paar', 'tagasi', 'möödus', 'Tallinna', 'saksa', 'gümnaasiumis', 'õppiva', 'Anna', 'vaba', 'aeg', 'proovides', ',', 'esinemistel', 'või', 'stuudioseinte', 'vahel', ',', 'siis', 'nüüd', 'leiab', 'teda', 'kas', 'hobusetallist', 'või', 'ratsutamast', '.']


In [13]:
print(homonym_dataset["labels"].value_counts())

labels
-         172100
sg g_S      2394
sg g_H      1946
sg n_S      1235
sg n_H       844
sg p_S       831
sg n_A       210
sg n_P       121
sg g_A        67
adt_S         66
sg g_P        35
sg n_N        35
adt_H         28
sg p_H        26
sg p_P        25
sg g_N        15
sg p_A         8
Name: count, dtype: int64


In [15]:
# Save the homonym dataset to a new parquet file
homonym_dataset.to_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_updated.parquet",
    index=False,
)

# Save the homonym sentences dataset to a new parquet file
homonym_sentences_dataset.to_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_updated_sentences.parquet",
    index=False,
)

<a id='further_preprocessing_31'></a>


#### Further preprocessing


In [ ]:
homonym_dataset = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_updated.parquet"
)

In [ ]:
test_size = 0.2
train_data, test_data, train_sources, test_sources = (
    HomonymPreprocessor.train_test_split(
        df=homonym_dataset,
        test_size=test_size,
        source_col="source",
        seed=seed,
        homonym_col="words",
        form_col="form",
    )
)

In [ ]:
# Print all unique homonym words in the dataset
# Only homonyms have labels
homonym_words = homonym_dataset[homonym_dataset["labels"] != "-"]["words"].unique()
print(f"Unique homonym words in the dataset: {homonym_words}")

Unique homonym words in the dataset: ['võitja' 'teooria' 'kampaania' ... 'seeniori' 'Pensionäri' 'Berendseni']


In [ ]:
# Count number of sentences in the dataset by grouping sentences by sentence_id and sources by source column and counting the number of unique sentence_id and source combinations
sentence_source_groups = homonym_dataset.groupby(["sentence_id", "source"])
num_sentences = sentence_source_groups.ngroups
print(f"Number of sentences in the dataset: {num_sentences}")

Number of sentences in the dataset: 7886


In [ ]:
print(
    f"Number of sentences in the train set: {train_data.groupby(['sentence_id', 'source']).ngroups}"
)
print(
    f"Number of sentences in the test set: {test_data.groupby(['sentence_id', 'source']).ngroups}"
)

Number of sentences in the train set: 3943
Number of sentences in the test set: 3943


In [50]:
display(homonym_dataset.head())
display(train_data.head())
display(test_data.head())

,sentence_id,words,form,pos,labels,infl_type,source
0,0,Edinburghi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,0,agulite,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,0,mehe,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,0,Irvine,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,0,Welshi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


,sentence_id,words,form,pos,labels,infl_type,source
0,0,Edinburghi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,0,agulite,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,0,mehe,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,0,Irvine,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,0,Welshi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


,sentence_id,words,form,pos,labels,infl_type,source
0,22,Kakskümmend,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,22,aastat,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,22,tagasi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,22,töötasin,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,22,teadurina,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [ ]:
# Save the train and test sets to new parquet files
train_data.to_parquet(
    HOMONYMS_DIRS["processed"] / f"homonyms_train_{100 - test_size * 100}.parquet",
    index=False,
)
test_data.to_parquet(
    HOMONYMS_DIRS["processed"] / f"homonyms_test_{test_size * 100}.parquet",
    index=False,
)

In [52]:
# Get unique homonymous words in the homonym_dataset and save the list to a txt file
# Each line in the txt file should contain one homonymous word
homonymous_words = homonym_dataset[homonym_dataset["labels"] != "-"]["words"].unique()
with open(
    HOMONYMS_DIRS["processed"] / "homonymous_words.txt", "w", encoding="utf-8"
) as f:
    for word in homonymous_words:
        f.write(f"{word}\n")

<a id='model_related_preprocessing_4'></a>


## Model related preprocessing (old)

These code blocks were used to preprocess the data into a mixed dataset for training and evaluation of the models. They are kept here for reference, but they are not used in the current version of the codebase, as mixed dataset preprocessing needs hyperparameter tuning to get the right proportions of the different datasets, and it is not yet clear what those proportions should be.


In [8]:
import json

from scripts.preprocessing.unique_labels import get_unique_labels

In [9]:
with open(OUTPUT_DIR / "unique_labels.json", "w", encoding="utf-8") as f:
    json.dump(get_unique_labels(), f, indent=2)
    print("Unique labels saved to unique_labels.json")

List of unique labels created
Unique labels saved to unique_labels.json


In [7]:
import pandas as pd

In [8]:
pq_enc = pd.read_parquet(
    ENC2017_DIRS["processed"] / "enc2017_morph_analysis_updated.parquet"
)
pq_ud_et_edt = pd.read_parquet(
    UD_ET_EDT_DIRS["processed"] / "ud_edt_morph_analysis_updated.parquet"
)
homonym_dataset = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall.parquet"
)

In [9]:
print(pq_enc.head())
print(pq_ud_et_edt.head())
print(homonym_dataset.head())

   sentence_id        words   form pos   labels         type  \
0            0  Teisipäeval  sg ad   S  sg ad_S  periodicals   
1            0       saabus      s   V      s_V  periodicals   
2            0   Iisraelist  sg el   H  sg el_H  periodicals   
3            0     Tallinna   sg g   H   sg g_H  periodicals   
4            0   lennujaama   sg g   S   sg g_S  periodicals   

                 source  
0  nc_10114_622631.json  
1  nc_10114_622631.json  
2  nc_10114_622631.json  
3  nc_10114_622631.json  
4  nc_10114_622631.json  
   sentence_id   words    form pos    labels   file_prefix  \
0            0     Aga           J         J  aja_ee199920   
1            0   mulle  sg all   P  sg all_P  aja_ee199920   
2            0  tundub       b   V       b_V  aja_ee199920   
3            0       ,           Z         Z  aja_ee199920   
4            0      et           J         J  aja_ee199920   

                   source  
0  et_edt-ud-dev_000.json  
1  et_edt-ud-dev_000.json  
2 

In [17]:
from scripts.preprocessing.common import sampler

sampler.main(
    [
        "--target-words",
        "100000",
        "--enc-path",
        str(ENC2017_DIRS["processed"] / "enc2017_morph_analysis_updated.parquet"),
        "--ud-path",
        str(UD_ET_EDT_DIRS["processed"] / "ud_edt_morph_analysis_updated.parquet"),
        "--hom-path",
        str(HOMONYMS_DIRS["processed"] / "homonyms_overall.parquet"),
        "--alpha",
        "0.4",
        "--beta",
        "0.5",
        "--gamma",
        "0.1",
        "--mode",
        "source",
        "--seed",
        "42",
        "--output-dir",
        "../data/sampled/",
    ]
)

Wrote sampled dataset to ..\data\sampled\sampled_dataset_20260311-162059.parquet
Wrote sampling report to ..\data\sampled\sampling_report_20260311-162059.json


<a id='end'></a>


### END
